In [1]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("metadata.csv")
results = pd.read_csv("./results.csv")

n_cases = len(df)

In [6]:
q_choices = {
    "prototypes_quality_choices": ["A", "B", "Both", "Neither"], # which shows the diagnosis?
    "prototypea_quality": ["Yes", "No"], # is A a good prototype of the diagnosis?
    "prototypeb_quality": ["Yes", "No"], # is B a good prototype of the diagnosis?
    "prototypes_quality_choices_2": ["A", "B", "Both", "Neither"], # which matches the test case?
    "explanation_a": ["Yes", "No"], # does A explain the test case?
    "explanation_b": ["Yes", "No"], # does B explain the test case?
}

make_col = lambda col, case_id: f"case{case_id}_{col}"
make_cols = lambda case_id: [make_col(c, case_id) for c in q_choices]
map_to_choice = lambda col, value: q_choices[col.split("_", maxsplit=1)[1]][value - 1]

In [7]:
for i in range(n_cases):
    for c in make_cols(i + 1):
        results[c] = results[c].apply(lambda value: map_to_choice(c, value))

In [ ]:
responses = []
for row_idx, row in results.iterrows():
    protossl_global_good = []
    protoecgnet_global_good = []
    protossl_case_good = []
    protoecgnet_case_good = []
    for i, protossl_a_or_b in enumerate(df["ProtoSSL Assignment"]):
        i = i + 1
        if protossl_a_or_b == "A":
            protossl_col = "prototypea_quality"
            protoecgnet_col = "prototypeb_quality"
        else:
            protoecgnet_col = "prototypea_quality"
            protossl_col = "prototypeb_quality"
        protossl_global_good.append(row[make_col(protossl_col, i)] == "Yes")
        protoecgnet_global_good.append(row[make_col(protoecgnet_col, i)] == "Yes")

        if protossl_a_or_b == "A":
            protossl_col = "explanation_a"
            protoecgnet_col = "explanation_b"
        else:
            protoecgnet_col = "explanation_a"
            protossl_col = "explanation_b"
        protossl_case_good.append(row[make_col(protossl_col, i)] == "Yes")
        protoecgnet_case_good.append(row[make_col(protoecgnet_col, i)] == "Yes")


    protossl_global_good = np.asarray(protossl_global_good)
    protoecgnet_global_good = np.asarray(protoecgnet_global_good)
    protossl_case_good = np.asarray(protossl_case_good)
    protoecgnet_case_good = np.asarray(protoecgnet_case_good)

    responses.append(
        {
            "ProtoSSL Global": protossl_global_good.sum() / n_cases,
            "ProtoECGNet Global": protoecgnet_global_good.sum() / n_cases,
            "ProtoSSL Case": protossl_case_good.sum() / n_cases,
            "ProtoECGNet Case": protoecgnet_case_good.sum() / n_cases,
        }
    )

In [11]:
print(pd.DataFrame(responses).to_string())

   ProtoSSL Global  ProtoECGNet Global  ProtoSSL Case  ProtoECGNet Case
0             1.00                0.90           1.00              0.90
1             0.80                0.70           0.80              0.60
2             0.80                0.45           0.55              0.50
3             0.95                0.60           0.90              0.55
4             0.95                0.60           0.95              0.75
5             0.95                0.65           0.90              0.80
6             0.95                0.75           0.70              0.65
